# TP2 - Parte 1: dataset y preprocesamiento

Notebook de documentación y tratamiento del dataset **70 Dog Breeds Image Dataset**: análisis
exploratorio, control de calidad, conjunto independiente de evaluación, preprocesamiento,
data augmentation y splits (ver `ENUNCIADO.md`, sección 3).

Tiene que poder leerse como un informe: celdas Markdown que explican qué se hace y por qué,
código y salidas (gráficos y tablas) **ejecutadas y guardadas**. Una notebook sin salidas
visibles se considera no entregada.

Lo que produce esta notebook lo reutilizan `2-train.ipynb` y `train_classifier`: los splits
finales (CSV/JSON con rutas y etiquetas), el conjunto independiente y el preprocesamiento
encapsulado en un módulo de `src/lib/`.

**Notebook en Google Colab (si corresponde):** <reemplazar por el link, con permiso de lectura para los docentes>

Si entregan la notebook en Colab, esta copia del repositorio funciona como notebook de referencia:
el link de arriba es obligatorio y la notebook de Colab debe estar ejecutada, con sus salidas
(ver `ENUNCIADO.md`, sección 8.2).

## Equipo
- Alumno 1 : Nicolas Duclos

## 1. Clonar el repositorio

Solo si trabajan en Colab. Si tu fork es privado, generá un token de acceso (GitHub → Settings →
Developer settings → Personal access tokens) y usalo en la URL, o subí un zip del proyecto a Colab/Drive.

In [1]:
# TO-DO: completar con la URL de tu fork.
# Repo privado: https://<TOKEN>@github.com/<usuario>/tuia-dog-recognition-app.git
#REPO_URL = "https://github.com/<usuario>/tuia-dog-recognition-app.git"

#!git clone $REPO_URL proyecto
#%cd proyecto

## 2. Instalar dependencias

Colab ya incluye torch, torchvision, opencv, numpy, scikit-learn y matplotlib;
solo se instala lo que falta.

In [2]:
#!pip install -q pydantic-settings python-dotenv kagglehub albumentations onnxruntime

## 3. Configuración del entorno

En Colab no hay PostgreSQL, por lo que se desactiva pgvector y se usa la base vectorial JSON.
La configuración se define por variables de entorno **antes** de importar `lib`.

In [ ]:
import os
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import torch

# Detectar raiz del proyecto independientemente de si Jupyter se lanza en / o /notebooks
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

os.environ.setdefault("USE_PGVECTOR", "false")

from lib.config import settings
from lib.preprocessing import (
    IMAGENET_MEAN,
    IMAGENET_STD,
    normalize_breed_name,
    bgr_to_rgb,
    get_eval_transforms,
    get_train_transforms,
    get_visual_augmentation_pipeline,
    preprocess_bgr_image,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATASET_DIR = (
    settings.dataset_path
    if settings.dataset_path.is_absolute()
    else ROOT / settings.dataset_path
)
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT        : {ROOT}")
print(f"DATASET_DIR : {DATASET_DIR} (existe={DATASET_DIR.exists()})")
print(f"IMAGE_SIZE  : {settings.image_size}")
print(f"GPU CUDA    : {torch.cuda.is_available()}")

## 4. Descargar el dataset

In [ ]:
!python scripts/download_dataset.py
#!ls data/dataset

Descargando gpiosenka/70-dog-breedsimage-data-set ...
Ya existe, se omite: C:\Users\nicod\Desktop\TUIA\Vision por Computadora\tuia-dog-recognition-app\data\dataset\dogs.csv
Ya existe, se omite: C:\Users\nicod\Desktop\TUIA\Vision por Computadora\tuia-dog-recognition-app\data\dataset\test
Ya existe, se omite: C:\Users\nicod\Desktop\TUIA\Vision por Computadora\tuia-dog-recognition-app\data\dataset\train
Ya existe, se omite: C:\Users\nicod\Desktop\TUIA\Vision por Computadora\tuia-dog-recognition-app\data\dataset\valid
Dataset disponible en C:\Users\nicod\Desktop\TUIA\Vision por Computadora\tuia-dog-recognition-app\data\dataset


## 5. Descripción del dataset

TO-DO:
- Cantidad de razas, imágenes por raza en cada split, formato, resolución y licencia de uso.
- Cómo están encuadradas las imágenes, comparado con los recortes de YOLO que va a recibir el
  clasificador en la Etapa 3.
- Limitaciones del dataset (razas parecidas, sesgos, etc.) y cómo pueden afectar los resultados.

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def scan_dataset(dataset_dir: Path) -> pd.DataFrame:
    records = []
    for split in ["train", "valid", "test"]:
        split_dir = dataset_dir / split
        if not split_dir.exists():
            continue
        for breed_dir in sorted(split_dir.iterdir()):
            if not breed_dir.is_dir():
                continue
            raw_breed = breed_dir.name
            breed = normalize_breed_name(raw_breed)
            for img_path in sorted(breed_dir.iterdir()):
                if img_path.is_file() and img_path.suffix.lower() in IMAGE_EXTS:
                    records.append(
                        {
                            "split": split,
                            "raw_breed_dir": raw_breed,
                            "breed": breed,
                            "filepath": str(img_path),
                            "rel_path": img_path.relative_to(DATA_DIR).as_posix(),
                            "ext": img_path.suffix.lower(),
                        }
                    )
    return pd.DataFrame(records)


df_raw = scan_dataset(DATASET_DIR)

# Verificar discrepancia de nombres de carpetas entre splits
dir_mismatches = df_raw[df_raw["raw_breed_dir"] != df_raw["breed"]][
    ["split", "raw_breed_dir", "breed"]
].drop_duplicates()
print("Carpetas con espacios irregulares:")
display(dir_mismatches)

summary_raw = (
    df_raw.groupby(["split", "breed"])
    .size()
    .unstack(fill_value=0)
    .reindex(["train", "valid", "test"])
)
print(f"\nTotal de razas normalizadas: {summary_raw.shape[1]}")
print("Total de imágenes por split:", df_raw["split"].value_counts().to_dict())
display(summary_raw.T.describe().round(2))

Carpetas con espacios irregulares:


,split,raw_breed_dir,breed
7976,valid,American Spaniel,American Spaniel



Total de razas normalizadas: 70
Total de imágenes por split: {'train': 7946, 'valid': 700, 'test': 700}


split,train,valid,test
count,70.00,70.0,70.0
mean,113.51,10.0,10.0
std,24.68,0.0,0.0
min,65.00,10.0,10.0
25%,99.00,10.0,10.0
50%,111.50,10.0,10.0
75%,125.00,10.0,10.0
max,198.00,10.0,10.0



- **Origen y licencia:** *70 Dog Breeds Image Dataset* (Kaggle, `gpiosenka/70-dog-breedsimage-data-set`), publicado bajo licencia **CC0: Public Domain**.
- **Estructura y volumen:** Contiene **70 clases** divididas originalmente en tres particiones (`train`, `valid`, `test`), en formato **JPEG** de 3 canales (RGB) y resolución estándar de **224 × 224 píxeles**.
- **Anomalía detectada en carpetas:** En la partición `valid`, la clase `American  Spaniel` contiene un doble espacio en el nombre del directorio respecto de `train/American Spaniel` y `test/American Spaniel`. Si no se normaliza el nombre de clase, `ImageFolder` desalinea los índices de clase en validación.
- **Comparación de encuadre con la Etapa 3 (YOLO):** Las imágenes del dataset son retratos centrados donde el animal ocupa la mayor parte del cuadro, con fondos relativamente limpios o desenfocados. En cambio, en la Etapa 3 (`DetectionService`), el clasificador recibirá recortes (`crop`) generados por las *bounding boxes* de YOLO sobre escenas reales: estos recortes pueden tener relaciones de aspecto diversas, márgenes de fondo, oclusiones parciales o múltiples perros cercanos.
- **Limitaciones y sesgos:**
  1. *Razas visualmente muy similares:* pares como `Cocker` / `American Spaniel` / `Blenheim`, `Labrador` / `Golden Retriever`, `Siberian Husky` / `Elk Hound` / `Coyote`, o `Bull Terrier` / `Pit Bull` presentan alta similitud inter-clase.
  2. *Cánidos silvestres mezclados con razas domésticas:* el dataset incluye `African Wild Dog`, `Coyote`, `Dhole` y `Dingo`.
  3. *Sesgo de fotografía de exposición/estudio:* predominan poses frontales y buena iluminación, lo que puede degradar el desempeño frente a fotos caseras con poca luz o ángulos extremos.

## 6. Análisis exploratorio (EDA)

TO-DO:
- Distribución de clases: imágenes por raza en cada split (gráfico y estadísticas). Desbalance.
- Resoluciones, relación de aspecto, canales, formatos y archivos corruptos.
- Grilla de ejemplos por raza, incluyendo pares de razas fáciles de confundir.
- Variaciones: iluminación, pose, fondo, oclusiones y cantidad de perros por imagen.

In [ ]:
# 1. Distribución de imágenes por raza y split
counts_per_split = (
    df_raw.groupby(["breed", "split"]).size().unstack(fill_value=0)[["train", "valid", "test"]]
)
counts_sorted = counts_per_split.sort_values("train", ascending=False)

train_min, train_max = counts_sorted["train"].min(), counts_sorted["train"].max()
imbalance_ratio = train_max / max(train_min, 1)
print(
    f"Train por raza -> min: {train_min} ({counts_sorted['train'].idxmin()}), "
    f"max: {train_max} ({counts_sorted['train'].idxmax()}), "
    f"ratio desbalance: {imbalance_ratio:.2f}x"
)
print(f"Valid por raza -> min: {counts_sorted['valid'].min()}, max: {counts_sorted['valid'].max()}")
print(f"Test  por raza -> min: {counts_sorted['test'].min()}, max: {counts_sorted['test'].max()}")

fig, ax = plt.subplots(figsize=(18, 5))
x = np.arange(len(counts_sorted))
ax.bar(x, counts_sorted["train"], width=0.8, label="train", color="#2b6cb0")
ax.axhline(
    counts_sorted["train"].mean(),
    color="#e53e3e",
    linestyle="--",
    linewidth=1.3,
    label=f"Media train ({counts_sorted['train'].mean():.1f})",
)
ax.set_xticks(x)
ax.set_xticklabels(counts_sorted.index, rotation=90, fontsize=8)
ax.set_ylabel("Cantidad de imágenes")
ax.set_title("Distribución de imágenes por raza en train (70 clases)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Control de calidad y filtrado

TO-DO: reglas explícitas y reproducibles (archivos corruptos, baja resolución, imágenes borrosas o
con exposición extrema, duplicados y casi-duplicados dentro y entre splits, imágenes mal etiquetadas).
Reportar cuántas imágenes descarta cada regla y mostrar ejemplos de descartes.

In [ ]:
## TO-DO: reglas de filtrado, conteo de descartes por regla y ejemplos.

## 8. Conjunto independiente de evaluación

TO-DO: imágenes que no pertenezcan al dataset (internet o fotos propias), con las mismas razas que
las carpetas del dataset. Incluir imágenes con un perro (Etapas 1 y 2), escenas con varios perros u
otros objetos e imágenes sin perros (Etapa 3). Documentar fuente y licencia; no subirlas al repositorio.

In [ ]:
## TO-DO: construcción y resumen del conjunto independiente (por ejemplo, en data/dataset/independent/<raza>/).

## 9. Preprocesamiento

TO-DO: definir y justificar el resize, la normalización y el orden de canales (la app entrega BGR).

⚠️ El mismo preprocesamiento debe usarse en las notebooks y en los servicios de la app:
encapsulalo en un módulo de `src/lib/` e importalo desde ambos lados.

In [ ]:
## TO-DO: transformaciones de preprocesamiento (importadas desde el módulo compartido).

## 10. Data augmentation

TO-DO: horizontal flip, rotación, blur, variaciones de brillo y contraste y ruido, con sus rangos
justificados. Mostrar una grilla de ejemplos aumentados de una misma imagen. Solo sobre `train`.

In [ ]:
## TO-DO: pipeline de data augmentation con ejemplos visuales.

## 11. Splits

TO-DO: definir train / validation / test (los del dataset o redefinidos, justificando), verificar
que no haya fuga de datos entre splits, fijar semillas y guardar los splits finales (CSV/JSON con
rutas y etiquetas) para `2-train.ipynb`.

In [ ]:
## TO-DO: splits finales, verificación de fuga de datos y exportación.

## 12. Resumen final

TO-DO: tabla con la cantidad de razas e imágenes por split, incluido el conjunto independiente.

In [ ]:
## TO-DO: tabla resumen.

## 13. Licencias y uso de imágenes

TO-DO: licencia del dataset, fuente y licencia de las imágenes del conjunto independiente y criterio
aplicado a imágenes con personas identificables.